# Visual Affect Behavioral Battery (cross-modal, multi-model)

**Claim.** Task-irrelevant **visual affect** systematically reorganizes a VLM's **decision policy**,
organized along an internal **valence/arousal** axis, **causally mediated** by an affect direction
**shared with text**, across a **battery** of decision constructs and **multiple model families**, with
**emotion-specific** (appraisal) structure.

**Novelty (deep-search 2026-08, docs/NOVELTY_SEARCH.md).** Prior work is either input-output VLM behavior
(no mechanism) or text-only mechanistic (no image). Closest: Sofroniew/Anthropic (text-only) and
2605.21980 (VLM emotion *recognition*, not decision). The cross-modal + decision-battery combination is open.

**Design.** For each model and each construct we measure a generation-free first-token option-logit under:
(1) **image arm** — distressing vs positive image (valence) and high vs low arousal;
(2) **steer arm** — steer the affect axis `+a` vs `-a` (causal upper bound) + a random-direction control;
(3) **mediation** — present a distressing image but RESTORE the affect projection to neutral: if the
behavioral shift disappears, it was mediated by the affect direction.
Plus **emotion-specificity** (fear vs anger, sad vs anxious) and an **emotion-perception-AFTER** control.

Score convention: **higher = more negative-affect-congruent** answer. Responsible-use: everyday judgments
scored by a first-token logit (a tendency); no generation, no judge. The affect->refusal gate is NOT run here.

## 0 · Install

In [ ]:
!pip -q install transformers accelerate pillow numpy scikit-learn

## 1 · Config

In [ ]:
import os, gc, contextlib, json, math
import numpy as np, torch
from PIL import Image

MODELS = ["google/gemma-3-12b-it", "Qwen/Qwen2.5-VL-7B-Instruct"]   # add LLaVA-OneVision / InternVL to go wide
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16
OUT_DIR= "/content/out"; os.makedirs(OUT_DIR, exist_ok=True)
ALPHA  = 0.008          # steering magnitude (fraction of local residual norm)
N_IMG  = 24             # images per affect group used per measurement
IMG_MAXDIM = 512
SEED   = 0
# OASIS: point these at the OASIS image folder + ratings csv (cols incl. Valence_mean, Arousal_mean)
OASIS_IMG = "/content/affect_data/oasis_images"
OASIS_CSV = "/content/affect_data/OASIS.csv"
print("config ready |", len(MODELS), "models | device", DEVICE)

## 1a · Hugging Face auth (gated models)

In [ ]:
try:
    from huggingface_hub import login
    _t = os.environ.get("HF_TOKEN")
    if not _t:
        try:
            from google.colab import userdata; _t = userdata.get("HF_TOKEN")
        except Exception: _t = None
    if _t: login(_t); print("HF auth ok")
    else: print("!! no HF_TOKEN found — set it if a gated model 401s")
except Exception as e: print("auth note:", e)

## 2 · Data — OASIS valence + arousal

OASIS gives **normative viewer** valence & arousal (Kurdi et al. 2017). We build valence tertiles
(`img_lo/mid/hi`) and an arousal split (`imgA_hi/imgA_lo`) taken within the mid-valence band so arousal
is not confounded with valence. If OASIS isn't present, this cell explains what to provide.

In [ ]:
import csv as _csv
def _load_img(p):
    im=Image.open(p).convert("RGB")
    if max(im.size)>IMG_MAXDIM:
        s=IMG_MAXDIM/max(im.size); im=im.resize((int(im.size[0]*s),int(im.size[1]*s)))
    return im

def load_oasis():
    if not (os.path.isdir(OASIS_IMG) and os.path.isfile(OASIS_CSV)):
        raise FileNotFoundError("Provide OASIS_IMG (folder of images) + OASIS_CSV (ratings with "
                                "Valence_mean, Arousal_mean, and an image filename/Theme column).")
    rows=[]
    with open(OASIS_CSV, newline="", encoding="utf-8", errors="ignore") as f:
        for r in _csv.DictReader(f):
            k={kk.strip(): vv for kk,vv in r.items()}
            # tolerant column lookup
            def g(*names):
                for n in names:
                    for kk in k:
                        if kk.lower()==n: return k[kk]
                return None
            theme=g("theme","file","filename","image")
            v=g("valence_mean","valence"); a=g("arousal_mean","arousal")
            if theme is None or v is None: continue
            # find the image file for this theme
            cand=[fn for fn in os.listdir(OASIS_IMG) if os.path.splitext(fn)[0].lower().startswith(str(theme).strip().lower())]
            if not cand: continue
            try: rows.append((os.path.join(OASIS_IMG,cand[0]), float(v), float(a) if a else float("nan")))
            except Exception: continue
    rows.sort(key=lambda x:x[1])
    n=len(rows); assert n>=60, f"too few OASIS rows parsed ({n})"
    t=n//3
    lo=[_load_img(p) for p,_,_ in rows[:t][:N_IMG]]
    hi=[_load_img(p) for p,_,_ in rows[-t:][-N_IMG:]]
    mid_rows=rows[t:2*t]
    mid=[_load_img(p) for p,_,_ in mid_rows[:N_IMG]]
    # arousal split within mid-valence band
    mid_by_a=sorted([r for r in mid_rows if not math.isnan(r[2])], key=lambda x:x[2])
    ta=max(4,len(mid_by_a)//3)
    loA=[_load_img(p) for p,_,_ in mid_by_a[:ta][:N_IMG]]
    hiA=[_load_img(p) for p,_,_ in mid_by_a[-ta:][-N_IMG:]]
    print(f"OASIS: {n} rated | valence lo/mid/hi = {len(lo)}/{len(mid)}/{len(hi)} | arousal lo/hi = {len(loA)}/{len(hiA)}")
    return dict(lo=lo, mid=mid, hi=hi, loA=loA, hiA=hiA)

IMGS = load_oasis()   # reuse across models

## 3 · Constructs, emotion texts, and the per-model runner

Each construct: `(name, prompt, NEG-congruent words, POS-congruent words)`, scored
`logsumexp(NEG) - logsumexp(POS)` — higher = more negative-affect-congruent.

In [ ]:
# ---- decision battery (task-irrelevant image is prepended as the visual prime) ----
BATTERY = [
 ("risk_taking",
  "Consider this proposal: a friend suggests a risky but exciting plan. In one word, do you take the risk?",
  ["No","Careful","Wait"], ["Yes","Sure","Go"]),                # higher = risk-averse
 ("trust",
  "You may send money to a stranger who could return it or keep it. In one word, do you trust them?",
  ["No","Withhold","Refuse"], ["Yes","Trust","Send"]),          # higher = less trust
 ("punishment_moral",
  "A colleague knowingly bends a minor rule to finish faster, causing slight inconvenience. In one word, is this acceptable?",
  ["No","Unacceptable","Wrong"], ["Yes","Acceptable","Fine"]),  # higher = harsher
 ("ambiguity_interp",
  "Your manager messages: 'We need to talk about your performance tomorrow.' In one word, this most likely means:",
  ["Bad","Negative","Ominous"], ["Good","Fine","Routine"]),     # higher = threat reading
 ("confidence",
  "You just gave an answer you are unsure about. In one word, how confident are you?",
  ["Low","Unsure","Doubtful"], ["High","Confident","Certain"]), # higher = less confident
]

# ---- text emotion vectors (Anthropic-style diff-in-means vs neutral); for cross-modal + specificity ----
EMO_TXT = {
 "desperation":["I am utterly desperate and out of options.","There's no way out; I'm panicking and helpless.","I'm at the end of my rope and afraid of what's next."],
 "fear":       ["I'm terrified; something is deeply wrong.","My heart is pounding with fear; I feel unsafe.","A cold dread grips me; danger feels close."],
 "anger":      ["I'm furious and want to lash out.","This is outrageous; I'm seething with anger.","I'm enraged and ready to retaliate."],
 "sadness":    ["A heavy sadness settles over me.","I feel low, grieving, and drained of hope.","Everything feels bleak and mournful."],
 "anxiety":    ["I'm on edge, restless, and full of worry.","My mind races with anxious what-ifs.","I feel tense and uneasy about everything."],
}
NEUTRAL_TXT=["The weather today is mild with light clouds.","This document summarizes the quarterly figures.","The train departs from platform four at noon.","The recipe calls for two cups of flour."]
# affect-perception probe (asked AFTER the battery): does the image read as negative or positive?
PERCEPT = ("In one word, the overall feeling of this image is:", ["Negative","Bad","Distressing"], ["Positive","Good","Pleasant"])

### 3a · `run_model(MODEL_ID)` — HF loader + reused helpers + directions + battery

Uses the HuggingFace path (works across Gemma-3, Qwen2.5-VL, LLaVA, InternVL): `output_hidden_states`
for the residual stream and `register_forward_hook` on the decoder layers for steering/ablation —
identical mechanics to the gate notebook's Tier-2 runner.

In [ ]:
from transformers import AutoProcessor
try: from transformers import AutoModelForImageTextToText as _AutoVLM
except Exception: from transformers import AutoModelForVision2Seq as _AutoVLM

def _find_decoder_layers(model):
    best=None
    for _,mod in model.named_modules():
        if isinstance(mod, torch.nn.ModuleList) and len(mod)>=8:
            child=mod[0]
            if any(("attn" in n.lower() or "attention" in n.lower()) for n,_ in child.named_modules()):
                best=mod
    return best

def run_model(MODEL_ID):
    R={"model":MODEL_ID}
    proc=AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    model=_AutoVLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map=DEVICE, trust_remote_code=True).eval()
    layers=_find_decoder_layers(model); nL=len(layers)
    dev=DEVICE; dt=DTYPE

    def bi(text, image=None):
        content=([{"type":"image"}] if image is not None else [])+[{"type":"text","text":text}]
        pr=proc.apply_chat_template([{"role":"user","content":content}], add_generation_prompt=True, tokenize=False)
        inp=proc(text=[pr], images=([image] if image is not None else None), return_tensors="pt")
        return {k:(v.to(dev) if torch.is_tensor(v) else v) for k,v in inp.items()}
    U=lambda v:(v/v.norm().clamp_min(1e-6)).to(dev,dt)
    def add_hook(vec,coef):
        u=U(vec)
        def h(m,i,o):
            if isinstance(o,tuple): return (o[0]+coef*u,)+tuple(o[1:])
            return o+coef*u
        return h
    def restore_hook(vec, target):        # push the projection onto `vec` to a fixed per-layer target
        u=U(vec).float()
        def h(m,i,o):
            H=o[0] if isinstance(o,tuple) else o; Hf=H.float()
            Hf=Hf+(target-(Hf@u)).unsqueeze(-1)*u
            return ((Hf.to(H.dtype),)+tuple(o[1:])) if isinstance(o,tuple) else Hf.to(H.dtype)
        return h
    @contextlib.contextmanager
    def hk(hooks):
        hd=[layers[l].register_forward_hook(h) for (l,h) in hooks]
        try: yield
        finally:
            for x in hd: x.remove()
    def RL(inp):
        with torch.no_grad(): out=model(**inp, output_hidden_states=True)
        hs=out.hidden_states[1:1+nL]
        return torch.stack([h.float()[0,-1].cpu() for h in hs])
    tok=proc.tokenizer if hasattr(proc,"tokenizer") else proc
    def idsof(ws):
        s=set()
        for w in ws:
            for pre in (" "+w, w):
                t=tok(pre, add_special_tokens=False).input_ids
                if t: s.add(t[0])
        return sorted(s)

    # ---- directions (diff-in-means, per layer) ----
    def axis(pos_imgs, neg_imgs, prompt="Describe what is happening in this image."):
        P=torch.stack([RL(bi(prompt,im)) for im in pos_imgs[:N_IMG]]).mean(0)
        Nn=torch.stack([RL(bi(prompt,im)) for im in neg_imgs[:N_IMG]]).mean(0)
        v=(P-Nn); return v/v.norm(dim=-1,keepdim=True).clamp_min(1e-6)
    a_val = axis(IMGS["lo"], IMGS["hi"])            # -> NEGATIVE valence (distress minus positive)
    a_aro = axis(IMGS["hiA"], IMGS["loA"])          # -> HIGH arousal
    _Nn = torch.stack([RL(bi(t)) for t in NEUTRAL_TXT]).mean(0)
    v_txt={}
    for e,st_ in EMO_TXT.items():
        m=torch.stack([RL(bi(t)) for t in st_]).mean(0); d=(m-_Nn); v_txt[e]=d/d.norm(dim=-1,keepdim=True).clamp_min(1e-6)
    cos=lambda u,w: float(np.mean([float(u[l]@w[l]) for l in range(nL)]))
    R["cos_val_desperation"]=cos(a_val, v_txt["desperation"])   # cross-modal alignment (image valence vs text emotion)

    # norm-scaled steering
    _probe=bi(BATTERY[0][1])
    with torch.no_grad(): _o=model(**_probe, output_hidden_states=True)
    norms=np.array([float(h[0,-1].float().norm()) for h in _o.hidden_states[1:1+nL]])
    st=lambda dirs,al:[(l, add_hook(dirs[l], al*norms[l])) for l in range(nL)]
    torch.manual_seed(SEED)
    RAND=torch.stack([torch.randn(a_val[l].shape) for l in range(nL)]); RAND=RAND/RAND.norm(dim=-1,keepdim=True).clamp_min(1e-6)

    def behav(prompt, image=None, hooks=(), A=None, B=None):
        inp=bi(prompt,image)
        with torch.no_grad(), hk(hooks): out=model(**inp)
        lp=torch.log_softmax(out.logits[0,-1].float(),-1)
        ida=[t for w in A for t in idsof([w])]; idb=[t for w in B for t in idsof([w])]
        return float(torch.logsumexp(lp[ida],0)-torch.logsumexp(lp[idb],0))
    def img_mean(prompt, imgs, A, B):
        return float(np.mean([behav(prompt, im, A=A, B=B) for im in imgs[:N_IMG]]))

    # per-layer clean a_val projection at the battery prompts (for the mediation restore target)
    def aval_target():
        vals=[]
        for (_,p,_,_) in BATTERY:
            with torch.no_grad(): o=model(**bi(p), output_hidden_states=True)
            hs=o.hidden_states[1:1+nL]
            vals.append([float(hs[l][0,-1].float() @ a_val[l].to(dev).float()) for l in range(nL)])
        return np.array(vals).mean(0)
    aclean=aval_target()

    # ---- the battery ----
    rows=[]
    for (name,p,A,B) in BATTERY:
        base   = behav(p, None, A=A, B=B)
        img_neg= img_mean(p, IMGS["lo"], A, B)          # distress image
        img_pos= img_mean(p, IMGS["hi"], A, B)          # positive image
        img_hiA= img_mean(p, IMGS["hiA"], A, B)         # high arousal
        img_loA= img_mean(p, IMGS["loA"], A, B)         # low arousal
        steer_neg= behav(p, None, hooks=st(a_val,+ALPHA), A=A, B=B)
        steer_pos= behav(p, None, hooks=st(a_val,-ALPHA), A=A, B=B)
        rand_neg = behav(p, None, hooks=st(RAND,+ALPHA), A=A, B=B)
        rand_pos = behav(p, None, hooks=st(RAND,-ALPHA), A=A, B=B)
        # mediation: distress image, but restore a_val projection to its neutral baseline
        med_rest = float(np.mean([behav(p, im, hooks=[(l, restore_hook(a_val[l], float(aclean[l]))) for l in range(nL)], A=A, B=B) for im in IMGS["lo"][:min(10,N_IMG)]]))
        rows.append(dict(name=name, base=base, img_neg=img_neg, img_pos=img_pos,
                         img_valence_effect=img_neg-img_pos, img_arousal_effect=img_hiA-img_loA,
                         steer_effect=steer_neg-steer_pos, rand_effect=rand_neg-rand_pos,
                         mediation_restored=med_rest))
        print("  %-16s base %+6.2f | img(neg-pos) %+6.2f | steer %+6.2f | rand %+6.2f | img->restore %+6.2f (vs base %+.2f)"
              %(name, base, img_neg-img_pos, steer_neg-steer_pos, rand_neg-rand_pos, med_rest, base))
    R["battery"]=rows

    # ---- emotion-specificity: fear vs anger, sad vs anxious on risk + trust ----
    spec=[]
    for (name,p,A,B) in [b for b in BATTERY if b[0] in ("risk_taking","trust")]:
        s={"construct":name}
        for e in ("fear","anger","sadness","anxiety"):
            s[e]=behav(p, None, hooks=st(v_txt[e],+ALPHA), A=A, B=B) - behav(p, None, A=A, B=B)
        spec.append(s)
        print("  specificity %-12s fear %+5.2f anger %+5.2f sad %+5.2f anx %+5.2f"%(name,s["fear"],s["anger"],s["sadness"],s["anxiety"]))
    R["specificity"]=spec

    # ---- emotion-perception-AFTER control: can the model label the image valence? ----
    pc_p, pc_A, pc_B = PERCEPT
    perc_neg=img_mean(pc_p, IMGS["lo"], pc_A, pc_B); perc_pos=img_mean(pc_p, IMGS["hi"], pc_A, pc_B)
    R["perception_gap"]=perc_neg-perc_pos    # >0 = model reads distress images as more negative
    print("  perception: distress=%+.2f positive=%+.2f (gap %+.2f)"%(perc_neg,perc_pos,perc_neg-perc_pos))

    del model; gc.collect()
    if DEVICE=="cuda": torch.cuda.empty_cache()
    return R

## 4 · Run all models

In [ ]:
ALL=[]
for mid in MODELS:
    print("\n=== %s ==="%mid)
    try:
        ALL.append(run_model(mid))
    except Exception as e:
        print("!! failed:", mid, "->", repr(e)); ALL.append({"model":mid,"error":str(e)})
    gc.collect(); torch.cuda.empty_cache() if DEVICE=="cuda" else None

## 5 · Save + susceptibility fingerprint

In [ ]:
import csv
json.dump(ALL, open(f"{OUT_DIR}/visual_affect_battery.json","w"), indent=2, default=float)
# per-model x construct image-valence effect = the "affective susceptibility fingerprint"
with open(f"{OUT_DIR}/visual_affect_fingerprint.csv","w",newline="") as f:
    w=csv.writer(f); w.writerow(["model","construct","base","img_valence_effect","img_arousal_effect","steer_effect","rand_effect","mediation_restored"])
    for R in ALL:
        if "battery" not in R: continue
        for r in R["battery"]:
            w.writerow([R["model"],r["name"],r["base"],r["img_valence_effect"],r["img_arousal_effect"],r["steer_effect"],r["rand_effect"],r["mediation_restored"]])
print("saved -> visual_affect_battery.json / visual_affect_fingerprint.csv")
for R in ALL:
    if "battery" not in R: continue
    npos=sum(r["img_valence_effect"]>0 for r in R["battery"])
    print("  %-34s %d/%d constructs move (predicted dir) under distress image | cos(img-valence,text-desperation)=%.2f"
          %(R["model"].split('/')[-1], npos, len(R["battery"]), R.get("cos_val_desperation",float("nan"))))
# to download: from google.colab import files; files.download(f"{OUT_DIR}/visual_affect_battery.json")